Number of nodes: 2708
Number of edges: 10556
Feature dimension: 1433
Number of classes: 7
Epoch: 020, Loss: 0.2842
Epoch: 040, Loss: 0.0783
Epoch: 060, Loss: 0.0474
Epoch: 080, Loss: 0.0537
Epoch: 100, Loss: 0.0427
Epoch: 120, Loss: 0.0313
Epoch: 140, Loss: 0.0374
Epoch: 160, Loss: 0.0254
Epoch: 180, Loss: 0.0263
Epoch: 200, Loss: 0.0323
Accuracy: 0.7960


In [ ]:
# %%
# SimaLab: Hands-on Graph Neural Networks (GNN) with PyTorch Geometric
# Tutorial 01: Node Classification on Cora with Graph Convolutional Networks (GCN)

# 1. Environment & Reproducibility Setup
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# Set seed for reproducibility
torch.manual_seed(42)


In [ ]:
# 2. Loading Cora Benchmark Dataset
# Loading the Cora dataset which is a citation network
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]

In [ ]:

# 3. Exploratory Graph Analysis
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of edges: {data.num_edges}")
print(f"Feature dimension: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")

In [ ]:


# 4. GCN Architecture Definition
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GCN, self).__init__()
        # First Graph Convolutional Layer
        self.conv1 = GCNConv(in_channels, hidden_channels)
        # Second Graph Convolutional Layer for node classification logits
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        # Neighborhood aggregation & message passing
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x


In [ ]:

# 5. Training Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN(dataset.num_features, 16, dataset.num_classes).to(device)
data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

# Training function
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()


In [ ]:

# 6. Training Execution
for epoch in range(1, 201):
    loss = train()
    if epoch % 20 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

In [ ]:

# 7. Evaluation & Latent Space Visualization
model.eval()
out = model(data.x, data.edge_index)
pred = out.argmax(dim=1)
acc = (pred[data.test_mask] == data.y[data.test_mask]).sum() / data.test_mask.sum()
print(f'Accuracy: {acc:.4f}')